# Demo LSTM Phân Tích Cảm Xúc Tiếng Việt
#### Nhóm 2:
- Võ Lê Ngọc Thịnh - 24521710
- Vũ Minh Phương - 24521421
- Nguyễn Hồng Phúc - 24521390
- Nguyễn Duy Khang - 24520755

Notebook này minh họa một pipeline NLP nhỏ dùng LSTM để phân loại review phim tiếng Việt thành `Tích cực` hoặc `Tiêu cực`. Dữ liệu trong demo được viết tay và có kích thước nhỏ, nên mục tiêu chính là hiểu quy trình xử lý văn bản và huấn luyện mô hình, không phải đạt độ chính xác production.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)

## 1. Tạo Dữ Liệu Demo

Ta tạo một dataset nhỏ gồm các câu review phim ngắn và nhãn cảm xúc tương ứng. Dataset được viết tay từng câu thay vì sinh bằng template để tránh việc train/test quá giống nhau.

- `label = 1`: review tích cực
- `label = 0`: review tiêu cực
- `train_df`: dùng để huấn luyện và tách validation
- `test_df`: giữ riêng để đánh giá cuối cùng

In [ ]:
train_reviews = [
    ("Phim rất tuyệt vời, tôi thích cách kể chuyện", 1),
    ("Diễn xuất tự nhiên và cảm xúc", 1),
    ("Cốt truyện hấp dẫn, càng xem càng cuốn", 1),
    ("Hình ảnh đẹp và âm nhạc rất hợp", 1),
    ("Tôi thấy phim đáng xem và khá cảm động", 1),
    ("Nhân vật chính được xây dựng tốt", 1),
    ("Phim nhẹ nhàng nhưng để lại nhiều cảm xúc", 1),
    ("Kết thúc trọn vẹn, không bị hụt hẫng", 1),
    ("Một bộ phim chỉn chu và có thông điệp rõ", 1),
    ("Tôi sẽ giới thiệu phim này cho bạn bè", 1),
    ("Nhiều cảnh hài duyên và không bị gượng", 1),
    ("Nhịp phim ổn, xem rất dễ chịu", 1),
    ("Phim rất tệ và nhàm chán", 0),
    ("Kịch bản rời rạc, tôi xem mà mất kiên nhẫn", 0),
    ("Diễn xuất gượng gạo và thiếu cảm xúc", 0),
    ("Cốt truyện dễ đoán, không có điểm nhấn", 0),
    ("Tôi thất vọng vì phim quá dài dòng", 0),
    ("Nhiều cảnh thừa làm mạch phim bị chậm", 0),
    ("Phim không đáng tiền vé", 0),
    ("Âm thanh khó chịu và lời thoại khá sáo", 0),
    ("Nhân vật nhạt, tôi không quan tâm tới câu chuyện", 0),
    ("Kết thúc vội vàng và thiếu thuyết phục", 0),
    ("Tôi thấy phim lộn xộn và mệt mỏi", 0),
    ("Xem xong không đọng lại gì", 0),
]

test_reviews = [
    ("Phim có nhiều đoạn rất hay và diễn viên diễn tốt", 1),
    ("Tôi thích thông điệp của phim, xem xong thấy vui", 1),
    ("Bộ phim hơi chậm nhưng vẫn ấm áp và đáng xem", 1),
    ("Hình ảnh đẹp, câu chuyện đơn giản nhưng dễ thương", 1),
    ("Phim buồn ngủ, nhiều cảnh kéo dài không cần thiết", 0),
    ("Tôi không thích phim này vì nội dung quá nhạt", 0),
    ("Diễn viên cố gắng nhưng kịch bản quá yếu", 0),
    ("Phim có ý tưởng tốt nhưng triển khai rất vụng", 0),
]

train_df = pd.DataFrame(train_reviews, columns=['text', 'label'])
test_df = pd.DataFrame(test_reviews, columns=['text', 'label'])

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df['sentiment'] = train_df['label'].map({1: 'Tích cực', 0: 'Tiêu cực'})
test_df['sentiment'] = test_df['label'].map({1: 'Tích cực', 0: 'Tiêu cực'})

print(f"Train samples: {len(train_df)}")
print(train_df['sentiment'].value_counts().to_string())
print(f"\nFinal test samples: {len(test_df)}")
print(test_df['sentiment'].value_counts().to_string())

display(train_df.reset_index(drop=True))
display(test_df.reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.countplot(data=train_df, x='sentiment', hue='sentiment', legend=False, ax=axes[0])
axes[0].set_title('Train set')
axes[0].set_xlabel('Nhãn')
axes[0].set_ylabel('Số câu')

sns.countplot(data=test_df, x='sentiment', hue='sentiment', legend=False, ax=axes[1])
axes[1].set_title('Final test set')
axes[1].set_xlabel('Nhãn')
axes[1].set_ylabel('Số câu')

plt.tight_layout()
plt.show()

## 2. Tiền Xử Lý Dữ Liệu

Mô hình LSTM không đọc trực tiếp được câu chữ, nên ta cần chuyển văn bản thành chuỗi số. `Tokenizer` học bộ từ vựng từ train set, sau đó biến mỗi câu thành một sequence token id.

Lưu ý: tokenizer chỉ được fit trên `train_df`. `test_df` chỉ được transform để mô phỏng dữ liệu mới mà mô hình chưa từng thấy.

In [ ]:
vocab_size = 300
max_len = 32
embedding_dim = 64

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

train_sequences = tokenizer.texts_to_sequences(train_df['text'])
test_sequences = tokenizer.texts_to_sequences(test_df['text'])

X_all = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
y_all = train_df['label'].values
X_test = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')
y_test = test_df['label'].values

word_index = tokenizer.word_index
print(f"Số từ khác nhau trong train corpus: {len(word_index)}")
print(f"Shape của X_all: {X_all.shape}")
print(f"Shape của X_test: {X_test.shape}")

example_id = 0
print("\nVí dụ train sau encoding và padding")
print("Text gốc:", train_df['text'].iloc[example_id])
print("Sequence:", train_sequences[example_id])
print("Padded:", X_all[example_id])
print("Label:", y_all[example_id], "=", train_df['sentiment'].iloc[example_id])

## 3. Chia Train/Validation/Test

Từ `train_df`, ta tách thêm một phần validation để theo dõi quá trình huấn luyện và dùng early stopping. Final test set được giữ riêng tới cuối notebook, tránh dùng test trong quá trình chọn mô hình.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all,
    test_size=0.25,
    random_state=SEED,
    stratify=y_all
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Train labels: {np.bincount(y_train)}  # [tiêu cực, tích cực]")
print(f"Val labels:   {np.bincount(y_val)}  # [tiêu cực, tích cực]")
print(f"Test labels:  {np.bincount(y_test)}  # [tiêu cực, tích cực]")

## 4. Xây Dựng Mô Hình LSTM

Mô hình gồm các lớp cơ bản cho phân loại cảm xúc nhị phân:

1. `Embedding`: biến token id thành vector dense.
2. `LSTM`: học thông tin theo thứ tự từ trong câu.
3. `Dense`: kết hợp đặc trưng đã học.
4. `Sigmoid`: xuất ra xác suất câu thuộc lớp tích cực.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    SpatialDropout1D(0.10),
    LSTM(48, dropout=0.10),
    Dense(32, activation='relu'),
    Dropout(0.20),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.build(input_shape=(None, max_len))
print("=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
model.summary()

## 5. Huấn Luyện Mô Hình

Mô hình được huấn luyện trên train set và theo dõi loss trên validation set. Vì dataset nhỏ, mô hình rất dễ học thuộc, nên dùng `EarlyStopping` để dừng khi validation loss không cải thiện thêm.

In [ ]:
print("MODEL INFORMATION")
print("=" * 60)
print(f"Vocab size: {vocab_size}")
print(f"Max sequence length: {max_len}")
print(f"Embedding dimension: {embedding_dim}")
print(f"LSTM units: 48")
print(f"Total parameters: {model.count_params():,}")

layer_rows = []
for layer in model.layers:
    layer_rows.append({
        'Layer': layer.name,
        'Type': layer.__class__.__name__,
        'Parameters': layer.count_params()
    })

display(pd.DataFrame(layer_rows))

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    batch_size=4,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=0
)

print(f"\nHuấn luyện hoàn thành sau {len(history.history['loss'])} epoch.")
print(f"Train accuracy cuối: {history.history['accuracy'][-1]:.2%}")
print(f"Validation accuracy tốt nhất: {max(history.history['val_accuracy']):.2%}")

## 6. Đánh Giá Mô Hình

Ở bước này, mô hình được đánh giá trên final test set. Ngoài accuracy, notebook in thêm classification report và confusion matrix để xem mô hình đang sai ở lớp nào.

Với dataset nhỏ, accuracy có thể dao động mạnh. Nếu sai một vài câu thì đó là điều bình thường và là điểm tốt để thảo luận về giới hạn của demo.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2%}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Tiêu cực', 'Tích cực'], digits=3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history.history['loss'], label='Train', marker='o')
axes[0].plot(history.history['val_loss'], label='Validation', marker='s')
axes[0].set_title('Loss qua từng epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary crossentropy')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train', marker='o')
axes[1].plot(history.history['val_accuracy'], label='Validation', marker='s')
axes[1].set_title('Accuracy qua từng epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Tiêu cực', 'Tích cực'],
    yticklabels=['Tiêu cực', 'Tích cực'],
    ax=axes[2]
)
axes[2].set_title('Confusion matrix')
axes[2].set_xlabel('Dự đoán')
axes[2].set_ylabel('Thực tế')

plt.tight_layout()
plt.show()

## 7. Dự Đoán Cảm Xúc Trên Dữ Liệu Mới

Sau khi đánh giá, ta thử đưa vào một vài câu review mới để xem mô hình dự đoán nhãn và xác suất tích cực như thế nào. Đây là phần dễ dùng khi trình bày demo trực tiếp.

In [ ]:
def predict_sentiment(text):
    """Dự đoán cảm xúc từ một đoạn văn bản tiếng Việt."""
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')
    positive_score = float(model.predict(padded, verbose=0)[0][0])
    label = 'Tích cực' if positive_score >= 0.5 else 'Tiêu cực'
    confidence = positive_score if label == 'Tích cực' else 1 - positive_score
    return label, confidence, positive_score

test_texts = [
    "Phim rất tuyệt vời, diễn xuất hay và câu chuyện cảm động",
    "Phim này tệ lắm, tôi thấy rất lãng phí thời gian",
    "Phim bình thường, có vài đoạn hay nhưng kết thúc hơi hụt",
    "Nhạc phim đẹp, hình ảnh chỉn chu và nội dung cuốn hút",
    "Kịch bản rời rạc, nhân vật nhạt và xem rất buồn ngủ",
]

rows = []
for text in test_texts:
    sentiment, confidence, positive_score = predict_sentiment(text)
    rows.append({
        'Câu review': text,
        'Dự đoán': sentiment,
        'Độ tin cậy': f"{confidence:.1%}",
        'Xác suất tích cực': f"{positive_score:.3f}",
    })

results_df = pd.DataFrame(rows)
display(results_df)